# 코스피 지수 데이터 수집

In [2]:
# --- common setup (run once) ---
import os
import time
import json
import math
import random
import datetime as dt
from typing import Optional, Dict, Any

import pandas as pd

# (권장) pykrx 설치
# !pip -q install pykrx

START = "2023-01-01"
END   = "2025-12-31"

KOSPI_INDEX_CODE = "1001"   # KOSPI 지수 코드
TICKER_OUT = "A1001"        # 요청하신: 티커 앞에 A 붙이기

OUT_DIR = "./kospi_features_csv"
os.makedirs(OUT_DIR, exist_ok=True)

def _sleep_jitter(base_sec: float = 0.5):
    time.sleep(base_sec + random.random() * base_sec)

def save_feature_csv(df: pd.DataFrame, filename: str) -> str:
    """
    df: 반드시 'date', 'ticker' 컬럼 포함
    """
    if "date" not in df.columns or "ticker" not in df.columns:
        raise ValueError("DataFrame must contain 'date' and 'ticker' columns.")
    df = df.sort_values(["date", "ticker"]).reset_index(drop=True)
    path = os.path.join(OUT_DIR, filename)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"[OK] saved: {path} rows={len(df):,}")
    return path

def ensure_date_column(df: pd.DataFrame, date_col_name: str = "date") -> pd.DataFrame:
    """
    pykrx는 인덱스가 날짜인 경우가 많아서, 컬럼 'date'로 빼주는 유틸
    """
    if isinstance(df.index, pd.DatetimeIndex) or df.index.name in ("날짜", "date"):
        df = df.copy()
        df[date_col_name] = df.index.astype("datetime64[ns]").date.astype(str)
        df = df.reset_index(drop=True)
        return df
    return df

print("OUT_DIR:", os.path.abspath(OUT_DIR))


OUT_DIR: c:\Users\OZ\OneDrive\AI공부\AI퀀트 과정 1기\프로젝트3\코스피분석\kospi_features_csv


## 1) 코스피 지수 OHLCV + 거래대금/거래량 (PyKrx)

지수의 일별 추이(종가/수익률 등) 만들 때 기본이 되는 파일

In [3]:
# --- 1) KOSPI index OHLCV (independent cell) ---
from pykrx import stock

def extract_kospi_index_ohlcv(start=START, end=END):
    df = stock.get_index_ohlcv_by_date(start.replace("-", ""), end.replace("-", ""), KOSPI_INDEX_CODE)
    # df columns (예상): 시가, 고가, 저가, 종가, 거래량, 거래대금 등 (KRX 제공 형식에 따라 달라질 수 있음)
    df = ensure_date_column(df, "date")

    out = df.copy()
    out["ticker"] = TICKER_OUT

    # 컬럼명 영문화(원하면 그대로 둬도 됨)
    rename_map = {
        "시가": "open",
        "고가": "high",
        "저가": "low",
        "종가": "close",
        "거래량": "volume",
        "거래대금": "value"
    }
    out = out.rename(columns={k: v for k, v in rename_map.items() if k in out.columns})

    cols = ["date", "ticker"] + [c for c in out.columns if c not in ("date", "ticker")]
    out = out[cols]
    return out

df_ohlcv = extract_kospi_index_ohlcv()
save_feature_csv(df_ohlcv, "kospi_index_ohlcv.csv")
df_ohlcv.head()


[OK] saved: ./kospi_features_csv\kospi_index_ohlcv.csv rows=731


코스피,date,ticker,open,high,low,close,volume,value,상장시가총액
0,2023-01-02,A1001,2249.95,2259.88,2222.37,2225.67,346344799,5200137586818,1759241799519040
1,2023-01-03,A1001,2230.98,2230.98,2180.67,2218.68,410245325,6149082624890,1753771077018843
2,2023-01-04,A1001,2205.98,2260.06,2198.82,2255.98,412841149,6487597995523,1783808765569816
3,2023-01-05,A1001,2268.20,2281.39,2252.97,2264.65,430977022,7521178466245,1791816776272757
4,2023-01-06,A1001,2253.40,2300.62,2253.27,2289.97,398606581,6764112169245,1811418923114710


## 2) 외국인/기관/개인 순매수(거래대금) — “시장단위 KOSPI” (PyKrx)

pykrx 문서 예시대로 "KOSPI"를 넣으면 시장 단위 투자자별 순매수(기본값) 를 일별로 뽑을 수 있습니다.

In [4]:
# --- 2) Investor net trading value (KOSPI market) ---
from pykrx import stock

def extract_kospi_investor_net_value(start=START, end=END):
    df = stock.get_market_trading_value_by_date(
        start.replace("-", ""),
        end.replace("-", ""),
        "KOSPI"  # market
        # on 파라미터: 기본은 순매수, 필요 시 on="매수"/"매도"
    )
    df = ensure_date_column(df, "date")
    out = df.copy()
    out["ticker"] = TICKER_OUT

    # 컬럼명을 분석-friendly하게 정리
    rename_map = {
        "기관합계": "inst_net_value",
        "기타법인": "other_corp_net_value",
        "개인": "retail_net_value",
        "외국인합계": "foreign_net_value",
        "전체": "total_net_value"
    }
    out = out.rename(columns={k: v for k, v in rename_map.items() if k in out.columns})

    cols = ["date", "ticker"] + [c for c in out.columns if c not in ("date", "ticker")]
    out = out[cols]
    return out

df_inv_net = extract_kospi_investor_net_value()
save_feature_csv(df_inv_net, "kospi_investor_net_trading_value.csv")
df_inv_net.head()


[OK] saved: ./kospi_features_csv\kospi_investor_net_trading_value.csv rows=731


,date,ticker,inst_net_value,other_corp_net_value,retail_net_value,foreign_net_value,total_net_value
0,2023-01-02,A1001,-266420384362,36849545095,222969530228,6601309039,0
1,2023-01-03,A1001,-355786681450,-10793617181,274820113950,91760184681,0
2,2023-01-04,A1001,-155571962,35423693015,-295934452202,260666331149,0
3,2023-01-05,A1001,-351706351490,13091171095,-164544528705,503159709100,0
4,2023-01-06,A1001,243733364898,6227820729,-564021112602,314059926975,0
